# 05. 앙상블

- 담당: **③ 안주형 (고성능 모델)** (이 파일은 담당자만 수정)
- 입력: `experiments/preds/` 에 `save_predictions()`로 저장된 모델들의 예측 확률
- 방법: 같은 행·같은 클래스 순서의 확률을 평균 → fold 밖 예측(oof)으로 Macro F1 비교

> 가중치를 oof 점수로 고르면 CV가 조금 낙관적으로 나와요. 가중치는 단순하게(동일 가중, 또는 몇 개 후보만) 유지해요.

## 0. 환경 설정

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-develop) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np, pandas as pd
from src.data_loader import load_data, split_xy, encode_target
from src.evaluation import macro_f1, per_class_f1, log_result, load_predictions
from src.prediction import make_submission

## 1. 데이터와 저장된 예측 불러오기

In [ ]:
train, test = load_data(DATA_DIR)
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)

preds = load_predictions(classes=le.classes_)   # 클래스 순서가 다르면 여기서 멈춰요
assert preds, "experiments/preds/ 에 저장된 예측이 없어요 → 각자 노트북에서 save_predictions()로 저장"

## 2. 모델별 점수 (oof 기준)

> oof_macro_f1은 5개 fold 예측을 합쳐 한 번에 계산한 값이라, fold별 점수의 평균(cv_mean)과 조금 달라요 (베이스라인: 0.3223 vs 0.3209). 모델끼리 비교할 때는 같은 방식끼리 비교해요.

In [ ]:
rows = []
for key, p in preds.items():
    assert p["oof"].shape == (len(y), len(le.classes_)), f"{key}: oof 크기가 달라요"
    assert p["test"].shape == (len(test), len(le.classes_)), f"{key}: test 크기가 달라요"
    rows.append({"model": key, "oof_macro_f1": round(macro_f1(y, p["oof"].argmax(1)), 4),
                 "cv_mean(기록)": p["meta"]["cv_mean"]})
pd.DataFrame(rows).sort_values("oof_macro_f1", ascending=False)

## 3. 앙상블 (동일 가중 평균)
`USE`에 섞을 모델 이름을 넣어요. 비어 있으면 전부 섞어요.

In [ ]:
USE = []          # 예) ["A_model__lgbm_v2", "P_light__logreg"]
WEIGHTS = None    # 예) [0.7, 0.3]  (None이면 동일 가중)

keys = USE or list(preds)
w = np.ones(len(keys)) / len(keys) if WEIGHTS is None else np.asarray(WEIGHTS) / np.sum(WEIGHTS)
oof_ens = sum(wi * preds[k]["oof"] for wi, k in zip(w, keys))
test_ens = sum(wi * preds[k]["test"] for wi, k in zip(w, keys))

ens_score = macro_f1(y, oof_ens.argmax(1))
print("섞은 모델:", keys, "| 가중치:", np.round(w, 3))
print(f"앙상블 oof Macro F1 = {ens_score:.4f}")
per_class_f1(y, oof_ens.argmax(1), le.classes_).head(10)

## 4. 기록과 제출 파일
제출은 팀장에게 CV 점수와 함께 알려 주세요.

In [ ]:
# log_result(owner="A_model", notebook="05_ensemble.ipynb", experiment="ens_" + "+".join(keys),
#            features="-", model="ensemble(mean)", params={"keys": keys, "weights": list(np.round(w, 3))},
#            cv={"mean": ens_score, "std": 0.0}, notes="oof 기준 점수")
# pred = le.inverse_transform(test_ens.argmax(1))
# make_submission(test, pred, "05_ensemble.csv")